# Turn 2D pose estimation into 3D

In [2]:
from mpl_toolkits.mplot3d import Axes3D
from pose_3d_projector import Pose3DProjector
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Set up notebook to display plots inline
%matplotlib inline

print("✓ Imports successful!")

✓ Imports successful!


## Setup file paths

The toml file was made using the freemocap software suite

In [3]:
# Path to your camera calibration TOML file
# This file contains the camera intrinsics and extrinsics
CALIBRATION_FILE = "/Users/robertseymour/Documents/recordings/2026-02-06_15-06-02/2026-02-06_14-54-43_camera_calibration.toml"

# Path to the 2D keypoint data (output from previous step)
# This is the data_2d folder created by the MediaPipe processor
KEYPOINTS_2D_DIR = "/Users/robertseymour/Documents/recordings/2026-02-06_15-06-02/mediapipe_output/data_2d"

# Path where you want to save the 3D outputs
OUTPUT_3D_DIR = "/Users/robertseymour/Documents/recordings/2026-02-06_15-06-02/data_3d"

# Path for visualizations
VISUALIZATION_DIR = "/Users/robertseymour/Documents/recordings/2026-02-06_15-06-02/visualizations"

print(f"Calibration file: {CALIBRATION_FILE}")
print(f"2D keypoints directory: {KEYPOINTS_2D_DIR}")
print(f"3D output directory: {OUTPUT_3D_DIR}")

# Verify files exist
calib_path = Path(CALIBRATION_FILE)
keypoints_path = Path(KEYPOINTS_2D_DIR)

if calib_path.exists():
    print("✓ Calibration file found")
else:
    print("✗ ERROR: Calibration file not found!")

if keypoints_path.exists():
    npy_files = list(keypoints_path.glob("*_keypoints.npy"))
    print(f"✓ Found {len(npy_files)} camera keypoint files")
    for f in npy_files:
        print(f"  - {f.name}")
else:
    print("✗ ERROR: 2D keypoints directory not found!")

# Create output directories
Path(OUTPUT_3D_DIR).mkdir(parents=True, exist_ok=True)
Path(VISUALIZATION_DIR).mkdir(parents=True, exist_ok=True)
print("✓ Output directories ready")

Calibration file: /Users/robertseymour/Documents/recordings/2026-02-06_15-06-02/2026-02-06_14-54-43_camera_calibration.toml
2D keypoints directory: /Users/robertseymour/Documents/recordings/2026-02-06_15-06-02/mediapipe_output/data_2d
3D output directory: /Users/robertseymour/Documents/recordings/2026-02-06_15-06-02/data_3d
✓ Calibration file found
✓ Found 3 camera keypoint files
  - camera_2_synchronized_keypoints.npy
  - camera_0_synchronized_keypoints.npy
  - camera_1_synchronized_keypoints.npy
✓ Output directories ready


## Initialsie the 3D projector

In [4]:
projector = Pose3DProjector(
    calibration_path=CALIBRATION_FILE,
    keypoints_dir=KEYPOINTS_2D_DIR,
    
    # Minimum number of cameras that must see a keypoint to triangulate it
    # 2 = minimum for triangulation
    # 3+ = more robust, better accuracy
    min_cameras_for_triangulation=2,
    
    # Minimum confidence threshold for using a 2D detection
    # Higher = stricter (only use high-confidence detections)
    # Lower = more permissive (use more detections but might include noise)
    confidence_threshold=0.3
)

print("\n✓ 3D Projector initialized!")
print(f"  - Cameras loaded: {len(projector.cameras)}")
print(f"  - Camera names: {list(projector.cameras.keys())}")
print(f"  - Min cameras required: 2")
print(f"  - Confidence threshold: 0.3")

2026-02-09 12:20:01,732 - INFO - Loaded calibration for 3 cameras
2026-02-09 12:20:01,735 - INFO - Loaded camera_2_synchronized: shape (1802, 33, 3)
2026-02-09 12:20:01,737 - INFO - Loaded camera_0_synchronized: shape (1802, 33, 3)
2026-02-09 12:20:01,739 - INFO - Loaded camera_1_synchronized: shape (1802, 33, 3)
2026-02-09 12:20:01,739 - INFO - Loaded 2D keypoints from 3 cameras



✓ 3D Projector initialized!
  - Cameras loaded: 3
  - Camera names: ['camera_0_synchronized', 'camera_1_synchronized', 'camera_2_synchronized']
  - Min cameras required: 2
  - Confidence threshold: 0.3


## Triangulate

In [5]:
# This performs Direct Linear Transform (DLT) triangulation on every frame

print("\n" + "="*70)
print("Starting 3D triangulation...")
print("="*70)

# This will triangulate all keypoints for all frames
# You'll see a progress bar
points_3d, metrics = projector.triangulate_all_frames()

print("\n✓ Triangulation complete!")
print(f"  - 3D points shape: {points_3d.shape}")
print(f"  - Format: (n_frames, n_keypoints, 3) where last dim is [x, y, z] in mm")


2026-02-09 12:20:38,103 - INFO - Triangulating frames 0 to 1802



Starting 3D triangulation...


Triangulating: 100%|██████████| 1802/1802 [00:01<00:00, 993.78it/s] 
2026-02-09 12:20:39,957 - INFO - Triangulation complete!



✓ Triangulation complete!
  - 3D points shape: (1802, 33, 3)
  - Format: (n_frames, n_keypoints, 3) where last dim is [x, y, z] in mm
